In [14]:
#This function uses Lemma 4.2 to prove Corollary 4.3. This is because if total number of copies is p(p-1), 
#then clearly, no m-invariant affine copy of the set exists.
def count_affine_copies(A, p):
    """This function counts the distinct affine copies of a set A in Z/pZ under the
    maps x -> a*x + b (a a unit, b arbitrary) and reports whether that count equals
    p*(p-1), the full size of the affine group."""
    A = [x % p for x in A]
    copies = set()
    # Apply every affine map; a runs over the units 1..p-1, b over all of Z/pZ.
    for a in range(1, p):
        for b in range(p):
            copies.add(tuple(sorted((a * x + b) % p for x in A)))
    count = len(copies)
    full = p * (p - 1)
    # count == full exactly when A has no nontrivial affine self-symmetry.
    return count, count == full

In [21]:
#This checks the total number of copies of an inputted set in Z/pZ, where right now p=31.
count_affine_copies({0, 1, 2, 3, 4, 6, 10, 11, 26}, 31)

(930, True)

In [1]:
"""
Fast resultant pipeline for MSTD multiplicative subgroups of order n in Z/pZ.
Now also reports the resultants.

Key speedups vs the naive O(n^4) version:
  1. Fix i = 0 (rotations only change Res by a sign -> same prime factors).
  2. Dedup collisions by class under the dihedral + side-swap group.
  3. Cache factorizations keyed on |Res|.
"""
import sys, time
from itertools import combinations
from sympy import symbols, Poly, cyclotomic_poly, resultant
from sympy.ntheory import factorint, isprime

sys.set_int_max_str_digits(10**7)
_X = symbols('x')
_PHI = {}


def canonical(i, j, k, l, n):
    """This function returns the orbit-minimum representative of a collision under
    rotation, reflection, and side-swap, so equivalent collisions (which share the
    same |Res|) collapse to one canonical tuple."""
    best = None
    for refl in (1, -1):                              # reflection e -> -e
        a0, b0, c0, d0 = (refl*i) % n, (refl*j) % n, (refl*k) % n, (refl*l) % n
        for t in range(n):                            # rotation by t
            a, b = sorted(((a0+t) % n, (b0+t) % n))
            c, d = sorted(((c0+t) % n, (d0+t) % n))
            # side-swap: order the two sides so the smaller pair comes first
            rep = (a, b, c, d) if (a, b) <= (c, d) else (c, d, a, b)
            if best is None or rep < best:
                best = rep
    return best


def _phi(n):
    """This function returns the nth cyclotomic polynomial as a sympy Poly, caching
    it so repeated runs for the same n reuse the same object."""
    if n not in _PHI:
        _PHI[n] = Poly(cyclotomic_poly(n, _X), _X)
    return _PHI[n]


def _order_n_element(p, n):
    """This function returns an element of order n in (Z/pZ)*, found by taking a
    primitive root to the power (p-1)/n."""
    pm1 = p - 1
    fac = list(factorint(pm1))
    # A primitive root is a unit that is not a proper-power residue for any prime
    # factor of p-1.
    g = next(c for c in range(2, p) if all(pow(c, pm1//q, p) != 1 for q in fac))
    return pow(g, pm1 // n, p)


def mstd_verdict(p, n):
    """This function builds the order-n multiplicative subgroup of (Z/pZ)* and
    returns (|A+A|, |A-A|, is_mstd) for it."""
    h = _order_n_element(p, n)
    A, e = [], 1
    for _ in range(n):
        A.append(e); e = (e * h) % p
    S = {(a + b) % p for a in A for b in A}
    D = {(a - b) % p for a in A for b in A}
    return len(S), len(D), len(S) > len(D)


def _status(msg):
    """This function prints a status line immediately (flushed), used for progress
    reporting during the long scan phases."""
    print(msg, file=sys.stdout, flush=True)


def enumerate_classes(n, period=10.0):
    """This function scans every candidate collision with i fixed to 0 and returns
    the list of distinct canonical classes, printing progress every `period`
    seconds."""
    nonzero = [e for e in range(n) if e != 0]
    total = n * (len(nonzero) * (len(nonzero) - 1)) // 2
    seen = set()
    classes = []
    t0 = time.time(); last = t0; scanned = 0
    for j in range(0, n):
        for k, l in combinations(nonzero, 2):
            scanned += 1
            if not ({k, l} == {0, j} or (0 == k and j == l)):
                can = canonical(0, j, k, l, n)
                # `classes` counts DISTINCT canonical representatives, not the
                # candidates scanned.  There are only finitely many collision
                # classes under the dihedral+swap group, and each class has many
                # members (all rotations/reflections/side-swaps of one collision)
                # that canonicalize to the same rep.  Once every class has been
                # discovered -- which happens well before the scan ends, since the
                # early candidates already cover all the reps -- every later
                # candidate collapses onto an already-seen rep, so len(classes)
                # plateaus while `scanned` keeps climbing toward `total`.
                if can not in seen:
                    seen.add(can)
                    classes.append(can)
            now = time.time()
            if now - last >= period:
                el = now - t0; rate = scanned / el
                eta = (total - scanned) / rate if rate else float('inf')
                _status(f"dedup {100*scanned/total:.0f}%  classes={len(classes)}  eta {eta:.0f}s")
                last = now
    _status(f"dedup done: {len(classes)} classes in {time.time()-t0:.0f}s")
    return classes


def run(n, verbose=True, period=10.0):
    """This function runs the full pipeline for order n: dedup collisions, compute
    each class's resultant, collect the primes p (with n | p-1) that divide any
    resultant, and give each an MSTD verdict, returning everything in a dict."""
    phi = _phi(n)
    factor_cache = {}
    algebraic = []
    prime_hits = {}
    res_by_class = []          # (class, R)            -- every class + its resultant
    prime_to_res = {}          # p -> set of |R| that p divides
    classes = enumerate_classes(n, period=period if verbose else 1e18)
    total = len(classes)
    t0 = time.time(); last = t0
    for nclass, (a, b, c, d) in enumerate(classes, 1):
        # Each collision becomes the polynomial x^a + x^b - x^c - x^d; its
        # resultant with Phi_n vanishes iff the collision holds among nth roots.
        g = Poly(_X**a + _X**b - _X**c - _X**d, _X)
        R = int(resultant(phi, g))
        res_by_class.append(((a, b, c, d), R))
        if R == 0:
            algebraic.append((a, b, c, d))
        else:
            aR = abs(R)
            # Cache which relevant primes divide each resultant value, since many
            # classes share the same |R|.
            if aR not in factor_cache:
                factor_cache[aR] = {int(p) for p in factorint(aR) if (p-1) % n == 0}
            for p in factor_cache[aR]:
                prime_hits[p] = prime_hits.get(p, 0) + 1
                prime_to_res.setdefault(p, set()).add(aR)
        now = time.time()
        if verbose and now - last >= period:
            el = now - t0; rate = nclass / el
            eta = (total - nclass) / rate if rate else float('inf')
            _status(f"calc {100*nclass/total:.0f}%  primes={len(prime_hits)}  eta {eta:.0f}s")
            last = now
    dt = time.time() - t0
    if verbose:
        _status(f"calc done: {total} classes in {dt:.0f}s")
    # Verify each candidate prime and run the MSTD check on its order-n subgroup.
    verified = []
    mstd = {}
    for p in sorted(prime_hits):
        if not isprime(p) or (p-1) % n != 0:
            continue
        verified.append(p)
        mstd[p] = mstd_verdict(p, n)
    distinct_res = sorted({abs(R) for _, R in res_by_class if R != 0})
    if verbose:
        hits = [p for p in verified if mstd[p][2]]
        _status(f"mstd done: {len(verified)} primes checked, MSTD hits={hits}")
        _status(f"distinct resultant values: {len(distinct_res)}")
    return dict(n=n, classes=total, seconds=dt, algebraic=algebraic,
                primes=verified, prime_hits=prime_hits, mstd=mstd,
                res_by_class=res_by_class, prime_to_res=prime_to_res,
                distinct_res=distinct_res)

In [2]:
#We now sweep odd orders n = 7, 9, ..., 141 to see if any MSTD subgroup of order n exists in some Z/pZ.
for x in range(3, 71):
    n = 2*x+1
    print(n)
    for a in sys.argv[1:]:
        if a.lstrip("-").isdigit():
            n = int(a); break
    r = run(n)
    print(f"n={r['n']}: {r['classes']} classes in {r['seconds']:.1f}s")
    print(f"algebraic (Res=0): {len(r['algebraic'])}")
    print(f"primes p with n|p-1 dividing some resultant: {len(r['primes'])}")
    print(f"max resultant value: {max(r['distinct_res'])}")
    print("===============================================================")
    #print(r['primes'][:60])

7
dedup done: 30 classes in 0s
calc done: 30 classes in 0s
mstd done: 2 primes checked, MSTD hits=[]
distinct resultant values: 5
n=7: 30 classes in 0.0s
algebraic (Res=0): 0
primes p with n|p-1 dividing some resultant: 2
max resultant value: 301
9
dedup done: 62 classes in 0s
calc done: 62 classes in 0s
mstd done: 4 primes checked, MSTD hits=[]
distinct resultant values: 10
n=9: 62 classes in 0.0s
algebraic (Res=0): 0
primes p with n|p-1 dividing some resultant: 4
max resultant value: 729
11
dedup done: 110 classes in 0s
calc done: 110 classes in 0s
mstd done: 6 primes checked, MSTD hits=[]
distinct resultant values: 9
n=11: 110 classes in 0.0s
algebraic (Res=0): 0
primes p with n|p-1 dividing some resultant: 6
max resultant value: 7513
13
dedup done: 177 classes in 0s
calc done: 177 classes in 0s
mstd done: 9 primes checked, MSTD hits=[]
distinct resultant values: 13
n=13: 177 classes in 0.0s
algebraic (Res=0): 0
primes p with n|p-1 dividing some resultant: 9
max resultant value: 355

KeyboardInterrupt: 

In [3]:
#This function allows us to check if for a given prime p, there exists an MSTD multiplicative subgroup in Z/pZ.
def mstd_subgroups(p):
    """This function checks, for a given prime p, every admissible order n (odd,
    7 <= n <= p**0.75, and n dividing p-1) and returns the list of such n whose
    order-n multiplicative subgroup of (Z/pZ)* is MSTD in Z/pZ."""
    bound = p ** 0.75
    hits = []
    # Sweep every odd n in [7, p**0.75] that divides p-1.
    for n in range(7, int(bound) + 1, 2):
        if (p - 1) % n != 0:
            continue
        # mstd_verdict(p, n) returns (|A+A|, |A-A|, is_mstd); keep n if MSTD.
        if mstd_verdict(p, n)[2]:
            hits.append(n)
    return hits

In [13]:
mstd_subgroups(4231)

[141]